# 職場の離職確率・離職者数シミュレーション（Colab版）

**使い方（3ステップ）**
1. 財務部から返送された `roster_form.csv`（人数を記入済み）を Google Drive の
   `ColabNotebooks/employee-attrition/workplace/` に置く
2. 下の設定セルの `ROSTER_CSV` をそのファイル名に合わせる
3. メニュー「ランタイム → すべてのセルを実行」

**出力**：①翌年度の離職確率（雇用形態別・年齢階級別）
②数年先までの従業員数指数（現在=100）の推移と離職者数（中途離職／定年の分解）
③ベアゼロ（定昇のみ）と通常昇給のシナリオ比較

物価などのマクロ情報は**自動で付与**されます（貼り付け作業は不要）。
データはご自身の Drive と Colab セッション内で完結します。

In [ ]:
# ── 設定（ここだけ書き換える）───────────────────────────────────────
DATA_MODE  = 'gdrive'   # 'gdrive' / 'local' / 'synthetic'（動作確認用）
GDRIVE_DATA_DIR = '/content/drive/MyDrive/ColabNotebooks/employee-attrition/data'
LOCAL_DATA_DIR  = '/path/to/JPSED/data'
ROSTER_CSV = '/content/drive/MyDrive/ColabNotebooks/employee-attrition/workplace/roster_form_example.csv'
REPO_URL   = 'https://github.com/dsk-yshkw/employee-attrition'

BASE_YEAR       = 2025   # 集計表の基準年
YEARS           = 5      # 何年先まで
N_SIMS          = 300    # モンテカルロ反復
RETIREMENT_AGE  = 65     # 定年（None で無効）
INDUSTRY_CODE   = 55     # 教育（大学等）
OCCUPATION_CODE = 70     # その他一般事務系職
WAGE_MODE       = 'fixed'   # 'fixed'=ベアゼロ（定昇のみ） / 'model'=学習した賃金モデル
TEISHO_RATE     = 0.017     # 定昇率（ベアゼロ時の名目昇給率）
FUTURE_INFLATION_PCT = 2.0  # 将来インフレ率の想定（%）

In [ ]:
# ── 環境準備（Colab: クローン＋インストール＋Drive マウント）──────────
import subprocess, sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('employee-attrition'):
        subprocess.run(['git','clone',REPO_URL], check=True)
    os.chdir('employee-attrition')
    subprocess.run([sys.executable,'-m','pip','install','-q','scikit-learn>=1.4','pandas','numpy','matplotlib'], check=True)
    if DATA_MODE=='gdrive':
        from google.colab import drive; drive.mount('/content/drive')
if os.path.basename(os.getcwd()) in ('notebooks','workplace'): os.chdir('..')
sys.path.insert(0, os.getcwd())
SYNTH = (DATA_MODE=='synthetic')
DATA_DIR = 'data/synthetic' if SYNTH else (GDRIVE_DATA_DIR if DATA_MODE=='gdrive' else LOCAL_DATA_DIR)
if SYNTH: ROSTER_CSV='workplace/roster_form_example.csv'
print('DATA_DIR:', DATA_DIR); print('ROSTER  :', ROSTER_CSV)

## 1. JPSED で較正済みモデルを学習（数分かかります）

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.data.loader import DataLoader
from src.data.panel_builder import PanelBuilder
from src.data.transitions import TransitionFrameBuilder
from src.data.macro import MacroData
from src.features.assembler import FeatureAssembler
from src.models.transitions import TransitionModels

pb  = PanelBuilder(DataLoader(DATA_DIR, synthetic=SYNTH))
fa  = FeatureAssembler(); mac = MacroData()
tf, FEATS = TransitionFrameBuilder(pb, fa).build()
tm = TransitionModels(FEATS).fit(tf)
print('学習完了: %d 遷移 / 全国離職率 %.3f' % (len(tf), tf['separated'].mean()))

## 2. 記入済み様式の読み込み → 合成コホート

- 「平均勤続」「平均年収」が空欄の行は **JPSED の同区分（雇用形態×年齢階級）の
  中央値で自動補完**します
- 給与台帳にない項目（学歴・家族等）は欠測のまま（モデルが欠測対応）
- 物価（CPI・インフレ率・実質年収）は**この場で自動付与**

In [ ]:
EMP_MAP={'正規':1,'契約':4,'パート':2,'派遣':3,'嘱託':5}
SEX_MAP={'男':1,'女':2}

form = pd.read_csv(ROSTER_CSV, encoding='utf-8-sig')
form.columns=['emp','sex','age_band','n','avg_tenure','avg_salary']
form['n']=pd.to_numeric(form['n'],errors='coerce').fillna(0).astype(int)
form=form[form['n']>0].copy()
form['age_lo']=form['age_band'].str.split('-').str[0].astype(float)
form['age_hi']=form['age_band'].str.split('-').str[1].astype(float)
print('記入行:', len(form), '/ 合計人数:', form['n'].sum())

# JPSED の同区分中央値（補完用）
ref=tf.copy(); ref['age_band_lo']=(ref['age']//5*5)
def cell_median(emp_code, age_lo, col):
    m=ref[(ref['contract_type']==emp_code)&(ref['age_band_lo']==age_lo)][col].median()
    if pd.isna(m): m=ref[ref['contract_type']==emp_code][col].median()
    if pd.isna(m): m=ref[col].median()
    return m

rng=np.random.default_rng(12345); rows=[]
for _,r in form.iterrows():
    emp=EMP_MAP[r['emp'].strip()]
    ten_avg = r['avg_tenure'] if pd.notna(r['avg_tenure']) else cell_median(emp, r['age_lo'], 'tenure_years')
    sal_avg = r['avg_salary'] if pd.notna(r['avg_salary']) else cell_median(emp, r['age_lo'], 'annual_income')
    for _ in range(int(r['n'])):
        age=rng.uniform(r['age_lo'], r['age_hi']+1)
        rows.append(dict(gender=SEX_MAP.get(r['sex'].strip(), np.nan), age=age,
            education=np.nan, has_spouse=np.nan, has_child=np.nan,
            num_children=np.nan, youngest_child_age=np.nan,
            contract_type=emp, industry=INDUSTRY_CODE, firm_size=np.nan,
            occupation=OCCUPATION_CODE, position=np.nan, weekly_hours=np.nan,
            is_regular=int(emp==1),
            tenure_years=float(np.clip(rng.normal(ten_avg,2), 0, max(age-18,0))),
            annual_income=float(max(sal_avg*rng.uniform(0.85,1.15), 50)),
            prev_annual_income=np.nan, salary_growth_rate=np.nan))
cohort0=pd.DataFrame(rows)

# マクロ自動付与（CPI 実績＋将来は想定インフレ率で延長）
cpi_hist=mac.lookup()['cpi_all_items']
CPI={}; last=float(cpi_hist.iloc[-1]); last_year=int(cpi_hist.index.max())
for y in range(int(cpi_hist.index.min()), BASE_YEAR+YEARS+1):
    if y in cpi_hist.index: CPI[y]=float(cpi_hist.loc[y]); last=CPI[y]
    elif y>last_year: last*= (1+FUTURE_INFLATION_PCT/100); CPI[y]=last
def infl(y): return (CPI[y]/CPI[y-1]-1)*100
ref0=BASE_YEAR-1
cohort0['cpi']=CPI[ref0]; cohort0['inflation_rate']=infl(ref0)
cohort0['real_annual_income']=cohort0['annual_income']/(CPI[ref0]/100)
cohort0['real_prev_annual_income']=np.nan; cohort0['real_salary_growth_rate']=np.nan
print('コホート構築完了:', len(cohort0), '人')

## 3. 翌年度の離職確率（現時点の構成に対する予測）

In [ ]:
cohort0['p_sep']=tm.p_separate(cohort0[FEATS])
inv_emp={v:k for k,v in EMP_MAP.items()}
by_emp=cohort0.assign(区分=cohort0['contract_type'].map(inv_emp)).groupby('区分').agg(
    人数=('p_sep','size'), 平均離職確率=('p_sep','mean'), 期待離職者数=('p_sep','sum'))
by_emp.loc['合計']=[len(cohort0), cohort0['p_sep'].mean(), cohort0['p_sep'].sum()]
display(by_emp.round(3))
band=(cohort0['age']//10*10).astype(int).astype(str)+'代'
by_age=cohort0.groupby(band).agg(人数=('p_sep','size'), 平均離職確率=('p_sep','mean'),
                                 期待離職者数=('p_sep','sum')).round(3)
display(by_age)

## 4. 数年シミュレーション：従業員数指数（現在=100）

各年：①定年退出（年齢≥定年、確定的）→②中途離職の抽選→③残留者の更新
（年齢+1・**勤続+1**・賃金ルール・実質化）。性別・家族構成は固定。

In [ ]:
def simulate_workplace(ci, years=YEARS, n_sims=N_SIMS, wage_mode=WAGE_MODE,
                       teisho=TEISHO_RATE, retirement_age=RETIREMENT_AGE, seed0=0):
    N0=len(ci); recs=[]
    for s in range(n_sims):
        rng=np.random.default_rng(seed0+s); st=ci.copy()
        for k in range(1, years+1):
            year=BASE_YEAR+k-1; ref=year
            retire = st['age']>=retirement_age if retirement_age is not None else pd.Series(False,index=st.index)
            n_ret=int(retire.sum()); st=st[~retire]
            if len(st)>0:
                p=tm.p_separate(st[FEATS]); q=rng.random(len(st))<p
                n_quit=int(q.sum()); st=st[~q]
            else: n_quit=0
            if len(st)>0:
                old=st['annual_income'].to_numpy(float)
                new=old*(1+teisho) if wage_mode=='fixed' else tm.wage_stay(st[FEATS])
                cpi=CPI[ref]
                st=st.assign(age=st['age']+1, tenure_years=st['tenure_years']+1,
                    prev_annual_income=old, annual_income=new,
                    salary_growth_rate=np.clip((new-old)/old,-1,3),
                    cpi=cpi, inflation_rate=infl(ref),
                    real_prev_annual_income=st['real_annual_income'],
                    real_annual_income=new/(cpi/100))
                st=st.assign(real_salary_growth_rate=np.clip(
                    (st['real_annual_income']-st['real_prev_annual_income'])
                    /st['real_prev_annual_income'],-1,3))
            recs.append(dict(sim=s,k=k,year=year+1,remain=len(st),quits=n_quit,retirements=n_ret))
    return pd.DataFrame(recs), N0

res,N0=simulate_workplace(cohort0.drop(columns=['p_sep']))
g=res.groupby('k')
summary=pd.DataFrame({'暦年':g['year'].first(),
    '従業員数指数（現在=100）':(g['remain'].mean()/N0*100),
    '5%点':(g['remain'].quantile(.05)/N0*100), '95%点':(g['remain'].quantile(.95)/N0*100),
    '中途離職（期待人数/年）':g['quits'].mean(), '定年退職（人数/年）':g['retirements'].mean()}).round(2)
print(f'初期人数 {N0} 人を 100 と基準化 / 賃金モード={WAGE_MODE}, 将来インフレ={FUTURE_INFLATION_PCT}%')
display(summary)

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4))
ks=summary.index.to_numpy()
ax[0].plot(np.r_[0,ks], np.r_[100,summary['従業員数指数（現在=100）']],'o-',color='#0072B2')
ax[0].fill_between(ks, summary['5%点'], summary['95%点'], alpha=.2, color='#0072B2')
ax[0].set_xlabel('years ahead'); ax[0].set_ylabel('headcount index (now=100)')
ax[0].set_title('Workforce index (90% band)')
ax[1].bar(ks-0.2, summary['中途離職（期待人数/年）'], width=.4, label='quits (model)', color='#D55E00')
ax[1].bar(ks+0.2, summary['定年退職（人数/年）'], width=.4, label='retirement (age rule)', color='#999999')
ax[1].set_xlabel('years ahead'); ax[1].set_ylabel('leavers / year'); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. シナリオ比較：ベアゼロ vs 通常昇給

In [ ]:
rows=[]
for mode,label in [('fixed','ベアゼロ（定昇のみ）'),('model','通常昇給（モデル賃金）')]:
    r,_=simulate_workplace(cohort0.drop(columns=['p_sep']), wage_mode=mode)
    m=r.groupby('k')['remain'].mean()/N0*100
    rows.append(pd.Series(m.values, index=m.index, name=label))
comp=pd.DataFrame(rows).T; comp.index.name='years ahead'
display(comp.round(2))
comp.plot(marker='o', figsize=(6,4), ylabel='headcount index (now=100)')
plt.title('Wage scenarios'); plt.tight_layout(); plt.show()

## 読み方の注意

- モデルは JPSED（全国パネル）で学習。職場固有の事情は雇用形態・年齢・勤続・
  賃金など**観測変数を通じてのみ**反映
- 出力は組織水準の**期待値と分布幅**。小規模組織では実現値のブレが大きい
- 定年は年齢ルールで確定処理（再雇用制度を模すなら嘱託転換に改修可）
- 補充採用は含まない（純減の推移）。採用計画を重ねる場合は毎年コホートに
  新規行を追加